# 线性回归的完整训练过程

Axiom 配套实验。请先读懂对应教材，再按三个阶段练习。计算在你自己的电脑运行。

## 准备环境

建议 Python 3.11 或 3.12，在单独虚拟环境中运行下面的安装命令，然后用 `jupyter notebook` 打开本文件。

```bash
python -m pip install notebook
python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
```

先运行“跟着做”。带 TODO 的函数初始返回 None，检查单元会提示待完成。补全后重新运行函数定义与检查单元。独立练习应另写实现，再核对结果。参考解答在文件末尾；网页的阶段勾选是个人学习记录，不代表系统已执行你的代码。

按《动手学深度学习》第 2 章和 3.1—3.3 节。只在 CPU 运行，数据自行生成。

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
torch.set_num_threads(1)
def make_data(weights,bias,seed=0):
    g=torch.Generator().manual_seed(seed)
    X=torch.randn(1200,2,generator=g)
    y=X@torch.tensor(weights,dtype=torch.float32)+bias+.01*torch.randn(1200,generator=g)
    return X[:1000],y[:1000,None],X[1000:],y[1000:,None]
def run_training(epoch_fn,weights=(2.,-3.4),bias=4.2,seed=0):
    torch.manual_seed(seed);Xt,yt,Xv,yv=make_data(weights,bias,seed)
    loader=DataLoader(TensorDataset(Xt,yt),batch_size=10,shuffle=True)
    model=nn.Linear(2,1);criterion=nn.MSELoss();optimizer=torch.optim.SGD(model.parameters(),lr=.03)
    for _ in range(10):epoch_fn(model,loader,criterion,optimizer)
    with torch.no_grad():mse=criterion(model(Xv),yv).item()
    error=max((model.weight.detach().ravel()-torch.tensor(weights)).abs().max().item(),abs(model.bias.item()-bias))
    return model,mse,error
def check_training(result):
    model,mse,error=result
    assert mse<.01 and error<.1,(mse,error)
    print('检查通过：测试 MSE=',mse,'最大参数误差=',error)

## 1. 跟着做

逐步运行示例，解释每个输入、运算和输出。再修改一个输入观察变化。

In [ ]:
def demo_epoch(model,loader,criterion,optimizer):
    for X,y in loader:
        optimizer.zero_grad()
        prediction=model(X)
        error=criterion(prediction,y)
        error.backward()
        optimizer.step()
result=run_training(demo_epoch)
check_training(result)
print('拟合参数：',result[0].weight.detach(),result[0].bias.detach())

## 2. 自己补全

在下面函数中填写 TODO。可回看示例，但请说明你填写的每一行做了什么。

In [ ]:
def student_epoch(model,loader,criterion,optimizer):
    for X,y in loader:
        # TODO: 清零、前向、计算损失、反向、更新
        pass
    # TODO: 全部补好后返回 True
    return None

In [ ]:
probe=nn.Linear(2,1);loader=DataLoader(TensorDataset(torch.ones(2,2),torch.ones(2,1)),batch_size=1)
if student_epoch(probe,loader,nn.MSELoss(),torch.optim.SGD(probe.parameters(),lr=.03)) is None:print('待完成：补全训练循环。')
else:check_training(run_training(student_epoch))

## 3. 独立完成

独立实现训练循环，用真实权重 [-1,0.5]、偏置 -2 和种子 7 重新生成数据、训练并测试。解释每一步的作用。

In [ ]:
def independent_epoch(model,loader,criterion,optimizer):
    # TODO: 独立写出训练循环；最后返回 True
    return None

In [ ]:
probe=nn.Linear(2,1)
if independent_epoch(probe,loader,nn.MSELoss(),torch.optim.SGD(probe.parameters(),lr=.03)) is None:print('待完成：独立训练循环。')
else:check_training(run_training(independent_epoch,weights=(-1.,.5),bias=-2.,seed=7))

## 提示与复盘

先检查输入形状、数据类型和边界情况。网页“查看提示”提供针对本实验的提示。记录你遇到的一处错误、修复方法和结果。通过数值检查后，还需要自己解释原理。

## 参考解答

先独立尝试再展开下方代码。示例是可核对的一种实现，不是唯一写法。

In [ ]:
check_training(run_training(demo_epoch,weights=(-1.,.5),bias=-2.,seed=7))